# Imports

In [91]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [48]:
# import dataset
df = pd.read_csv('../data/kaggle_b2_fraud_train_v3.csv')

# 1. COMPRÉHENSION INITIALE DES DONNÉES

## 1.1 Vue d'ensemble

In [52]:
df.shape

(160000, 56)

### Valeurs abérantes

In [53]:
import pandas as pd

# Définition de règles de plausibilité pour certaines colonnes
rules = {
    'age': lambda x: x >= 0,
    'tenure_months': lambda x: x >= 0,
    'annual_income_eur': lambda x: x >= 0,
    'credit_score': lambda x: (x >= 300) & (x <= 850),
    'num_transactions_30d': lambda x: x >= 0,
    'avg_amount_30d_eur': lambda x: x >= 0,
    'max_amount_30d_eur': lambda x: x >= 0,
    'tx_amount_total_30d_eur': lambda x: x >= 0,
    'max_to_avg_ratio': lambda x: x >= 0,
    'days_since_last_login': lambda x: x >= 0,
    'support_tickets_90d': lambda x: x >= 0,
    'chargebacks_12m': lambda x: x >= 0,
    'failed_payments_6m': lambda x: x >= 0,
    'num_devices_30d': lambda x: x >= 0,
    'chargeback_resolution_time_days': lambda x: x >= 0,
}

# Stocker le nombre d'erreurs
impossible_counts = {}

for col, rule in rules.items():
    if col in df.columns:
        mask_invalid = ~df[col].isna() & ~rule(df[col])
        count_invalid = mask_invalid.sum()
        if count_invalid > 0:
            impossible_counts[col] = count_invalid

# Affichage
if impossible_counts:
    print("Colonnes avec valeurs impossibles et nombre d'erreurs :")
    for col, count in impossible_counts.items():
        print(f"{col}: {count} erreur(s)")
else:
    print("Aucune valeur impossible détectée selon les règles définies.")

Colonnes avec valeurs impossibles et nombre d'erreurs :
age: 84 erreur(s)
tenure_months: 203 erreur(s)
annual_income_eur: 117 erreur(s)
avg_amount_30d_eur: 71 erreur(s)


In [61]:
# suppression des âges négatifs
df = df[df['age'] >= 0]
# suppression des dates de création de compte négatives
df = df[df["tenure_months" ]>= 0]
# suppression des revenus négatifs
df = df[df["annual_income_eur"] >= 0]
# suppression des revenus négatifs
df = df[df["avg_amount_30d_eur"] >= 0]

In [60]:
df.iloc[:, 0:50].head(2)

,customer_id,account_id,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,...,tx_amount_total_30d_eur,max_to_avg_ratio,internal_signal_1,internal_signal_2,internal_signal_3,internal_signal_4,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8
0,CUST_6O9Q8D4I36,ACC_TXXXTNEUVKFY,34,108,38635.01,544.0,20,60.92,80.16,4.9,...,1218.40,1.3158,-0.99355,-1.34156,-0.68676,-1.54627,0.39006,0.10963,0.55097,-0.56104
1,CUST_FGUGTW230C,ACC_70VD7A4FFWCW,48,2,19912.97,703.0,21,112.11,571.12,0.3,...,2354.31,5.0943,-0.44874,0.23573,-0.17429,-0.00054,0.03265,-0.40256,0.36218,0.86583


### Typologie de données

In [86]:
# is_new_device → convertir en int64 (ou bool si 0/1)
df['is_new_device'] = df['is_new_device'].astype('int64')

# signup_source → garder comme object (string)
df['signup_source'] = df['signup_source'].astype('object')

# postal_code → convertir en string pour garder les codes avec zéro en début
df['postal_code'] = df['postal_code'].astype('str')

# days_since_last_login → convertir en int si tu veux uniquement des jours entiers
df['days_since_last_login'] = df['days_since_last_login'].astype('int64')

# 2. ANALYSE DE LA TARGET (Variable Cible)

## 2.1 Distribution et déséquilibre

In [97]:
# Comptage absolu
df['target_is_fraud'].value_counts(normalize=True)*100

# ==> Rééchantillonnage ou Pondération des classes

target_is_fraud
0    96.917408
1     3.082592
Name: proportion, dtype: float64

## 2.2 Analyse temporelle de la target (si applicable)

In [101]:
# S'assurer que signup_date est au format datetime
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')

# Grouper par mois (ou jour) et calculer le taux de fraude
fraud_rate_time = df.groupby(df['signup_date'].dt.to_period('M'))['target_is_fraud'].mean()

# Optionnel : afficher le taux en %
fraud_rate_time_percent = fraud_rate_time * 100
print(fraud_rate_time_percent.head(10))

## la fraude reste relativement stable au fil du temps, 
#sans tendance forte à la hausse ou à la baisse.

# aucune variable temporelle explicite n’est nécessaire pour corriger un biais saisonnier, 
# mais il faut garder signup_date si l’on veut éventuellement inclure 
# des features temporelles pour le modèle (ex. mois d’inscription, saison).

signup_date
2024-01    3.626244
2024-02    3.314528
2024-03    3.269391
2024-04    2.976400
2024-05    3.510515
2024-06    2.951838
2024-07    3.339395
2024-08    2.577320
2024-09    3.075848
2024-10    2.798764
Freq: M, Name: target_is_fraud, dtype: float64


### corrélation avec la feature ( dataleakage)

In [105]:
# Sélection des colonnes numériques (excluant la target)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [col for col in num_cols if col != 'target_is_fraud']

# Calcul de la corrélation de Pearson avec la target
corr_target = df[num_cols + ['target_is_fraud']].corr()['target_is_fraud'].sort_values(ascending=False)

print("Corrélation avec la target :")
print(corr_target.head(7))


## chargeback_resolution_time_days    0.676291
## post_event_status_code             0.640706
# ==> les supprimer lors du preprocess

Corrélation avec la target :
target_is_fraud                    1.000000
chargeback_resolution_time_days    0.676291
post_event_status_code             0.640706
num_devices_30d                    0.067326
ip_risk_z                          0.062932
tx_amount_total_30d_eur            0.049607
avg_amount_30d_eur                 0.046779
Name: target_is_fraud, dtype: float64


# 3 Valeurs manquantes : 

## 3.1 Quantification

In [112]:
(df.isna().mean() * 100).sort_values(ascending=False).to_frame(name='percent_missing').head(13)

,percent_missing
partner_risk_indicator,97.013470
legacy_partner_score,96.142459
secondary_email,92.126487
region,28.281706
credit_score,4.990215
max_amount_30d_eur,4.945768
device_trust_z,4.002351
customer_note,3.005886
ip_risk_z,2.964306
occupation,2.963590


### Missing par ligne : combien de lignes ont 0, 1, 5, 10+ valeurs manquantes ?

In [109]:
missing_per_row = df.isna().sum(axis=1)
missing_per_row.value_counts().sort_index()

0       11
1      489
2    10424
3    71800
4    45743
5     9796
6     1153
7       72
8        5
Name: count, dtype: int64

In [119]:
colonnes = [
    'partner_risk_indicator', 'legacy_partner_score', 'secondary_email', 
    'region', 'credit_score', 'max_amount_30d_eur', 'device_trust_z', 
    'customer_note', 'ip_risk_z', 'occupation', 'last_ticket_subject', 
    'merchant_category'
]

# Calcul du taux de fraude et de la différence
résumé = []
for col in colonnes:
    taux = df.groupby(df[col].isna())['target_is_fraud'].mean()
    diff = taux.get(True, 0) - taux.get(False, 0)  # missing - non-missing
    résumé.append({
        'diff_missing_present': diff *100
    })

résumé_df = pd.DataFrame(résumé).sort_values('diff_missing_present', key=abs, ascending=False)
print(résumé_df)

    diff_missing_present
10             -0.784577
5               0.706518
11             -0.476671
8               0.362245
7               0.288838
9               0.288313
6              -0.225786
3              -0.212463
4              -0.144841
0               0.059897
1               0.055558
2              -0.053754


In [ ]:
# Les colonnes last_ticket_subject (-0,78) et max_amount_30d_eur (0,71) portent un signal fort de fraude, mais il est en grande partie corrélé au fait qu’un client passe peu ou aucune commande. Il est donc plus efficace de créer directement une feature nb_commands qui capture cette information, et de supprimer les deux colonnes, qui contiennent en plus beaucoup de valeurs manquantes.

# 4. DOUBLONS

## 4.1 doublons stricts

In [121]:
df.duplicated().sum()

np.int64(6)

In [127]:
dup_customer = df[df.duplicated('customer_id', keep=False)].sort_values('customer_id')
dup_account  = df[df.duplicated('account_id', keep=False)].sort_values('account_id')

print(f"Duplicatas customer_id : {dup_customer.shape[0]}")
print(f"Duplicatas account_id  : {dup_account.shape[0]}")

Duplicatas customer_id : 3780
Duplicatas account_id  : 3780


In [130]:
mask_dup_customer = df['customer_id'].duplicated(keep=False)

# Appliquer le filtre pour ne garder que ces lignes
df_dup_customer = df[mask_dup_customer]



In [134]:
df_dup_customer.sort_values(by="customer_id", ascending=False)

,customer_id,account_id,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,...,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8,terms_accepted_flag,partner_risk_indicator,manual_review_result,post_event_status_code,chargeback_resolution_time_days,legacy_partner_score
55879,CUST_ZYIJNM1EAE,ACC_BIPLM79Q3I99,51,19,51165.36,646.0,27,42.68,241.63,4,...,0.50171,1.23267,0.54669,-0.53420,1,NaN,approve,0,0.0,NaN
60334,CUST_ZYIJNM1EAE,ACC_BIPLM79Q3I99,51,19,51165.36,646.0,27,42.68,241.63,4,...,0.50171,1.23267,0.54669,-0.53420,1,NaN,approve,0,8.4,NaN
12515,CUST_ZY78Y3MGQM,ACC_RQSDUVNTRYDN,39,13,60848.96,684.0,34,77.67,511.12,1,...,1.74233,-0.55847,1.41284,1.45052,1,NaN,approve,0,4.5,NaN
92561,CUST_ZY78Y3MGQM,ACC_RQSDUVNTRYDN,39,13,60848.96,684.0,34,77.67,511.12,1,...,1.74233,-0.55847,1.41284,1.45052,1,NaN,approve,0,2.2,NaN
38890,CUST_ZXBYUS9JLP,ACC_NKD67L2RDF5U,35,0,53184.31,626.0,23,26.22,139.43,7,...,-0.86671,0.57078,-0.61417,0.77689,1,NaN,approve,0,4.1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121294,CUST_00KH8AI10R,ACC_BUZ0K7NP06KI,52,10,25546.70,734.0,29,90.59,500.33,2,...,-0.37613,-1.54773,1.18771,0.14252,1,NaN,approve,0,0.0,NaN
64385,CUST_00IFKUFWR2,ACC_APJ9PF0K82OE,36,5,30477.93,697.0,31,66.64,422.94,24,...,2.49757,0.57274,-0.90293,0.61389,1,NaN,block,4,40.3,NaN
104198,CUST_00IFKUFWR2,ACC_APJ9PF0K82OE,36,5,30477.93,697.0,31,66.64,422.94,24,...,2.49757,0.57274,-0.90293,0.61389,1,NaN,block,4,32.5,NaN
109169,CUST_00B2T3J07Q,ACC_WVKOBDZNH09W,42,7,13718.99,637.0,22,19.17,51.29,31,...,-1.79131,0.38407,-1.38315,0.30033,1,NaN,block,4,29.7,NaN


In [146]:
# Conserver une ligne aléatoire par customer_id
df_clean = df.groupby('customer_id').sample(n=1, random_state=42).reset_index(drop=True)

print(f"Lignes avant : {df.shape[0]}, lignes après fusion : {df_clean.shape[0]}")

Lignes avant : 139493, lignes après fusion : 137603


# Variables numériques : 

In [158]:
# Sélectionner les variables numériques
num_cols = df.select_dtypes(include='number').columns

# Matrice de corrélation
corr_matrix = df[num_cols].corr()


# Identifier les corrélations fortes (>0.9)
strong_corr = [(col1, col2, corr_matrix.loc[col1, col2])
               for col1 in num_cols for col2 in num_cols
               if col1 != col2 and abs(corr_matrix.loc[col1, col2]) > 0.7]

# Affichage
print("Variables fortement corrélées (>0.9) :")
for col1, col2, val in strong_corr:
    print(f"{col1} ↔ {col2} : corr = {val:.2f}")

Variables fortement corrélées (>0.9) :
annual_income_eur ↔ income_log : corr = 0.76
annual_income_eur ↔ income_estimate_alt_eur : corr = 0.82
credit_score ↔ credit_score_norm : corr = 1.00
avg_amount_30d_eur ↔ max_amount_30d_eur : corr = 0.79
avg_amount_30d_eur ↔ tx_amount_total_30d_eur : corr = 0.94
max_amount_30d_eur ↔ avg_amount_30d_eur : corr = 0.79
max_amount_30d_eur ↔ tx_amount_total_30d_eur : corr = 0.74
income_log ↔ annual_income_eur : corr = 0.76
income_log ↔ income_estimate_alt_eur : corr = 0.93
income_estimate_alt_eur ↔ annual_income_eur : corr = 0.82
income_estimate_alt_eur ↔ income_log : corr = 0.93
credit_score_norm ↔ credit_score : corr = 1.00
tx_amount_total_30d_eur ↔ avg_amount_30d_eur : corr = 0.94
tx_amount_total_30d_eur ↔ max_amount_30d_eur : corr = 0.74


### Outliers

In [154]:
# Sélectionner variables numériques
num_cols = df.select_dtypes(include='number').columns.tolist()
num_cols.remove('target_is_fraud')  # enlever la target si elle est numérique

# Dictionnaire pour stocker le taux d'outliers par variable
outlier_stats = {}

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    # Flag outlier
    df[col + '_outlier'] = ((df[col] < lower) | (df[col] > upper))
    
    # Taux d'outliers par classe
    taux_par_classe = df.groupby('target_is_fraud')[col + '_outlier'].mean() * 100
    outlier_stats[col] = taux_par_classe

# Affichage
for col, taux in outlier_stats.items():
    print(f"{col}: \n{taux}\n")

age: 
target_is_fraud
0    0.400908
1    0.372093
Name: age_outlier, dtype: float64

tenure_months: 
target_is_fraud
0    4.50837
1    3.00000
Name: tenure_months_outlier, dtype: float64

annual_income_eur: 
target_is_fraud
0    4.298299
1    3.000000
Name: annual_income_eur_outlier, dtype: float64

credit_score: 
target_is_fraud
0    0.746340
1    0.697674
Name: credit_score_outlier, dtype: float64

num_transactions_30d: 
target_is_fraud
0    0.823267
1    1.116279
Name: num_transactions_30d_outlier, dtype: float64

avg_amount_30d_eur: 
target_is_fraud
0    2.985362
1    6.162791
Name: avg_amount_30d_eur_outlier, dtype: float64

max_amount_30d_eur: 
target_is_fraud
0    4.242823
1    7.093023
Name: max_amount_30d_eur_outlier, dtype: float64

days_since_last_login: 
target_is_fraud
0    5.009135
1    6.395349
Name: days_since_last_login_outlier, dtype: float64

support_tickets_90d: 
target_is_fraud
0    4.743589
1    5.186047
Name: support_tickets_90d_outlier, dtype: float64

chargebac

In [157]:
# Dictionnaire pour stocker la différence
diff_outliers = {}

for col in num_cols:
    taux = df.groupby('target_is_fraud')[col + '_outlier'].mean()
    diff = taux.get(1,0) - taux.get(0,0)  # fraude - non-fraude
    diff_outliers[col] = diff

# Convertir en DataFrame et trier par valeur absolue croissante
diff_df = pd.DataFrame({
    'variable': diff_outliers.keys(),
    'diff_fraud_nonfraud': diff_outliers.values()
})

diff_df['abs_diff'] = diff_df['diff_fraud_nonfraud'].abs()
diff_df = diff_df.sort_values('abs_diff', ascending=False).reset_index(drop=True)

print(diff_df[['variable','diff_fraud_nonfraud','abs_diff']])

                           variable  diff_fraud_nonfraud  abs_diff
0            post_event_status_code             0.925092  0.925092
1   chargeback_resolution_time_days             0.853540  0.853540
2                     is_new_device             0.066317  0.066317
3                   num_devices_30d             0.057006  0.057006
4                            is_vpn             0.051412  0.051412
5           tx_amount_total_30d_eur             0.033468  0.033468
6                avg_amount_30d_eur             0.031774  0.031774
7                max_amount_30d_eur             0.028502  0.028502
8                   chargebacks_12m             0.023729  0.023729
9                     tenure_months            -0.015084  0.015084
10            days_since_last_login             0.013862  0.013862
11          income_estimate_alt_eur            -0.013598  0.013598
12                annual_income_eur            -0.012983  0.012983
13               failed_payments_6m             0.005457  0.00

# Variables catégorielles 

In [159]:
df.select_dtypes(include='object').nunique().sort_values(ascending=False)

customer_id             137603
account_id              137603
referrer_code           137603
postal_code              73376
secondary_email          10846
city                        81
region                      16
country                     13
occupation                  11
last_ticket_subject         10
customer_note               10
payment_method              10
merchant_category            9
signup_source                6
os                           6
browser                      6
device_type                  5
channel                      4
plan_type                    4
manual_review_result         3
dtype: int64

In [166]:
cols = ['device_type','channel','plan_type','browser']

for c in cols:
    stats = df.groupby(c).agg(
        fraud_rate=('target_is_fraud','mean'),
        pct=('target_is_fraud','count')
    )
    stats['pct'] = stats['pct'] / stats['pct'].sum() * 100
    print(f"\n=== {c} ===")
    print(stats.sort_values('fraud_rate', ascending=False))


=== device_type ===
             fraud_rate        pct
device_type                       
phone          0.031149  51.920168
desktop        0.030853  20.981698
laptop         0.030291  17.986566
tablet         0.030166   8.103632
iot_device     0.028450   1.007936

=== channel ===
             fraud_rate        pct
channel                           
partner_api    0.035226   1.994365
web            0.030924  47.940040
mobile_app     0.030749  44.016546
call_center    0.029154   6.049049

=== plan_type ===
            fraud_rate        pct
plan_type                        
enterprise    0.032586   2.045981
basic         0.031585  45.053157
standard      0.030163  37.931652
premium       0.029979  14.969210

=== browser ===
         fraud_rate        pct
browser                       
Firefox    0.034661   9.968959
Other      0.031700   0.995032
Safari     0.030905  25.098750
Opera      0.030466   2.000100
Chrome     0.030417  55.032152
Edge       0.028239   6.905006
